# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Record sets found: {len(metadata.record_sets)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their field @ids

if not metadata.record_sets:
    print("No record sets found in the metadata.")
else:
    for rs in metadata.record_sets:
        print(f"RecordSet: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        if 'fields' in rs:
            print(f"  Fields and their @id:")
            for field in rs['fields']:
                field_id = field['@id']
                field_name = field.get('name', 'N/A')
                dtype = field.get('dataType', 'N/A')
                print(f"    - {field_id}: {field_name} [{dtype}]")
        else:
            print("  No fields listed.")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set

# Use the record set IDs obtained from previous cell:
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame with {len(dataframes[record_set_id])} records, columns: {dataframes[record_set_id].columns.tolist()}")
        else:
            print(f"No records found for {record_set_id}.")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")
    print()
# Pick the first available RecordSet with data for EDA example
if dataframes:
    selected_rs = record_set_ids[0]
    print(f"Using {selected_rs} for further analysis.")
    print(dataframes[selected_rs].head())
else:
    selected_rs = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if selected_rs and selected_rs in dataframes:
    df = dataframes[selected_rs]
    print("Available columns:", df.columns.tolist())
    # Find a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try to convert any column to numeric if not natively detected
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                if pd.api.types.is_numeric_dtype(df[col]):
                    numeric_field = col
                    break
            except Exception:
                continue
    if numeric_field:
        threshold = df[numeric_field].quantile(0.75) if np.issubdtype(df[numeric_field].dtype, np.number) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a groupable field
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() < 20:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped data by {group_field} (showing mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs and selected_rs in dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[selected_rs][numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in {selected_rs}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot if suitable second numeric column exists
    candidate_cols = [col for col in dataframes[selected_rs].columns if col != numeric_field]
    scatter_field = None
    for col in candidate_cols:
        if pd.api.types.is_numeric_dtype(dataframes[selected_rs][col]):
            scatter_field = col
            break
    if scatter_field:
        plt.figure(figsize=(6, 4))
        plt.scatter(dataframes[selected_rs][numeric_field], dataframes[selected_rs][scatter_field], alpha=0.6)
        plt.xlabel(numeric_field)
        plt.ylabel(scatter_field)
        plt.title(f"{numeric_field} vs. {scatter_field}")
        plt.show()
    else:
        print("No second numeric field for scatter plot.")
else:
    print("Visualization not available due to missing data.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the dataset using the Croissant schema and explored its structure using `mlcroissant`.
- Key record sets, their fields, and the dataset structure were listed by their `@id` as per the Croissant specification.
- A representative numeric field was selected to filter, normalize, and group/tabulate the data, demonstrating typical preprocessing steps.
- Visualizations gave quick insights into the data's distribution.

This workflow can be adapted for further domain-specific analyses relevant to extracellular vesicle proteomics data.